# BGE-M3 Top-1·2·3·5 + Mapping-end 정식 비교

READY 39건을 BGE-M3로 한 번만 Top-5까지 검색하고, 각 K에서 Mapping-end 공식 메타/실제 API 검증과 골드 평가를 수행합니다. Colab 런타임은 GPU로 설정하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/gold-v1-lock'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
READY_CSV = DRIVE_ROOT / 'inputs' / 'gold_measurement_scopefix_kosis_ready.csv'
GOLD_CSV = DRIVE_ROOT / 'inputs' / 'gold_measurement_v1_locked.csv'
RUN_DIR = DRIVE_ROOT / 'runs' / 'bge_topk_mapping_end'
REUSE_BASE = False  # 중단 후 재실행할 때 True

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
import sys

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-ml.txt'], check=True)
print('repo:', REPO_DIR)

In [ ]:
from google.colab import userdata

for path, label in [(READY_CSV, 'READY 39 CSV'), (GOLD_CSV, 'locked gold CSV'), (INDEX_DIR / 'manifest.json', 'BGE index')]:
    if not path.exists():
        raise FileNotFoundError(f'{label}가 없습니다: {path}')
if not os.environ.get('KOSIS_API_KEY'):
    os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY')
if not os.environ.get('KOSIS_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 KOSIS_API_KEY를 등록하세요.')
print('inputs and API key: ready')

In [ ]:
TABLE_INDEX = REPO_DIR / 'kosis_table_summary.csv'
command = [
    sys.executable, str(REPO_DIR / 'run_kosis_topk_experiment.py'),
    '--input', str(READY_CSV),
    '--gold', str(GOLD_CSV),
    '--table-index', str(TABLE_INDEX),
    '--semantic-index', str(INDEX_DIR),
    '--out-dir', str(RUN_DIR),
    '--ks', '1', '2', '3', '5',
    '--semantic-top-k', '50',
    '--rerank-top-k', '20',
    '--device', 'cuda',
    '--delay', '0.05',
]
if REUSE_BASE:
    command.append('--reuse-base')
subprocess.run(command, check=True)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

summary = pd.read_csv(RUN_DIR / 'topk_summary.csv', encoding='utf-8-sig')
display(summary)
display(Markdown((RUN_DIR / 'topk_report.md').read_text(encoding='utf-8')))